In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from tqdm.auto import tqdm
import pandas as pd
import torch
import itertools
from statistics import mode


In [2]:
val_data = []
with open("dev.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    val_data.append(json.loads(line))


test_data = []
with open("test.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    test_data.append(json.loads(line))

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200")
model = AutoModelForSeq2SeqLM.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200", device_map="auto",  torch_dtype=torch.float16)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [4]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    # return set(text.split())
    cleaned = []
    for word in text.split():
        if word.endswith("."):
            word= word[:-1]
        cleaned.append(word)
    return set(cleaned)
            
    


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)

    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))


        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option


    if  not best_option:
        
        best_option = None
        max_overlap = 0
        new_candidate = []
        for z in candidate_words :
            new_candidate.extend( list(z))
        for option in options:
            option_words = preprocess(option)

            new_options =[]
            for z in option_words :
                new_options.extend( list(z))

            overlap = len( set(list(new_candidate)).intersection( set(list(new_candidate))) )

            if overlap > max_overlap:
                max_overlap = overlap
                best_option = option

        return best_option, options.index(best_option)+1

            
    return best_option, options.index(best_option)+1

In [5]:
mannaaa = 'abc'


In [6]:
from string import Template
prompt_template= Template('''Youre a question answering expert. Please answer the question using the context and abbreviations provided. Only provided the correct option (A, B, C, D, E, F, G or H):\n
$question
context: $context      
                                                        
$options
''')


# prompt_template2= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
# $question
# context: $context      
                                                        
# $options
# $question
# ''')


In [7]:

# data[0]
k = 20


submission = {"answers":[]}
option_header = ["option A ", "option B ", "option C ", "option D ", "option E ","option F ", "option G ", "option H " ]
map_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
pbar = tqdm(range(len(val_data)))
for example in val_data:
    options = []
    opts = []
    context = ''
    for i, sample in enumerate(example['question']['choices']):
        # print(key)
        
    
        options.append(option_header[i]+ sample['text'])
        opts.append((sample['text'], option_header[i].split("option")[1]))
        context += '\n'+ sample['para']

    combined_fact = example['combinedfact']

    # combined_fact = example["fact1"] + '\n' + example["fact2"]
    # combined_fact = example["fact1"] + '\n' + example["fact2"]
# 

    all_permutations = list(itertools.permutations(opts))
    all_permutations = random.sample(all_permutations, k if len(all_permutations)>k else len(all_permutations))
    # print(all_permutations)
    batch_prompts = []
    batch_options = []
    option_maps = []
    batch_options_text = []
    for option_set in all_permutations:
        options  = []
        option_text = []
        option_map = {}
        for i in range(len(option_header)):
            options.append(option_header[i] + option_set[i][0])
            option_text.append(option_set[i][0])
            option_map[i+1]  = option_set[i][1]
        option_maps.append(option_map)
        batch_options.append(options)
        batch_options_text.append(option_text)
        batch_prompts.append(prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options)))



    input  = tokenizer(batch_prompts, return_tensors="pt").to("cuda")
    # print(input)
    out = model.generate(**input,  max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    texts = tokenizer.batch_decode(out, skip_special_tokens=True)

    answers = []
    for text, options, option_map in zip(texts, batch_options_text, option_maps):
        # print(text)
        # text = text.split("<pad>")[1][1:-4]
        # print(text)

  
       
        ans, id = choose_most_likely_option(options, text)

        answers.append(option_map[id])

    

    

    # answer_only = text

    

    submission["answers"].append(mode(answers))
    pd.DataFrame(submission).to_csv("t5k20_qasc_f1f2.csv")
    
    pbar.set_description(f"Pred: {id:.4f}")
    pbar.update(1)
pbar.close()
# print(answer_only)

  0%|          | 0/926 [00:00<?, ?it/s]

2024-11-12 15:21:24.413033: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-12 15:21:24.446187: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-12 15:21:25.108343: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
